# Building and Querying an Inverted Index

The inverted index is the fundamental data structure behind every search engine.
Instead of scanning every document for query terms, we flip the structure:
map each term to the documents that contain it. This demo builds one from scratch
and evaluates Boolean queries using efficient pointer-based merge algorithms.

**Learning goals:**
- Build an inverted index from a document collection step by step
- Understand the three core structures: vocabulary, postings lists, document table
- Evaluate Boolean queries (AND, OR, NOT) using streaming merge with two pointers
- See why sorted postings enable sub-linear query evaluation

**Prerequisites:** Chapter 1 (Boolean retrieval concepts, term frequency)

In [1]:
from shared.synthetic_collection import MINI, documents, text, ids
from shared.text import tokenize
from shared.display import print_table, display_md

## Why Inversion?

A **forward index** maps each document to its terms. To answer a query, we must
scan every document and check whether it contains our terms.

In [2]:
# Forward index: document -> terms
forward_index = {}
for doc_id, doc_text in documents():
    forward_index[doc_id] = tokenize(doc_text)

# Show the forward index for a few documents
rows = [[doc_id, " ".join(tokens[:8]) + "..."] for doc_id, tokens in list(forward_index.items())[:4]]
print_table(rows, headers=["Doc ID", "Terms (first 8)"])

| Doc ID   | Terms (first 8)                                        |
|:---------|:-------------------------------------------------------|
| b1       | the cat and dog in the forest a...                     |
| b2       | cats of the woodland wild cats begin an...             |
| b3       | the forest hound a loyal dog follows a...              |
| b4       | feline detective a clever cat solves mysteries with... |

In [3]:
# To find documents containing "cat", we must scan ALL documents
query_term = "cat"
matches = []
scan_count = 0

for doc_id, tokens in forward_index.items():
    scan_count += 1
    if query_term in tokens:
        matches.append(doc_id)

display_md(
    f"**Forward index scan for** `{query_term}`\n\n"
    f"- Documents scanned: **{scan_count}** (all of them!)\n"
    f"- Matches found: **{len(matches)}** -> {matches}\n\n"
    f"Cost: O(N x avg_doc_length) = O({len(forward_index)} x {sum(len(t) for t in forward_index.values()) // len(forward_index)})"
)

**Forward index scan for** `cat`

- Documents scanned: **12** (all of them!)
- Matches found: **5** -> ['b1', 'b4', 'b9', 'b10', 'b12']

Cost: O(N x avg_doc_length) = O(12 x 13)

With 10,000 documents averaging 200 terms each, a single query forces us to inspect
2,000,000 term positions. The inverted index solves this by pre-computing the answer
for every possible single-term query.

## Building the Inverted Index

The construction process has three steps: tokenize each document, build the
vocabulary, and create sorted postings lists with term frequency.

In [4]:
# Step 1: Tokenize all documents
doc_tokens = {}
for doc_id, doc_text in documents():
    doc_tokens[doc_id] = tokenize(doc_text)

display_md(f"**Tokenized** {len(doc_tokens)} documents from the MINI collection.")

**Tokenized** 12 documents from the MINI collection.

In [5]:
# Step 2: Build the inverted index
# Structure: {term: [(doc_id, tf), (doc_id, tf), ...]}
inverted_index = {}

for doc_id, tokens in doc_tokens.items():
    # Count term frequencies in this document
    term_counts = {}
    for token in tokens:
        term_counts[token] = term_counts.get(token, 0) + 1

    # Add to postings lists
    for term, tf in term_counts.items():
        if term not in inverted_index:
            inverted_index[term] = []
        inverted_index[term].append((doc_id, tf))

# Sort postings by doc_id for efficient merge operations
for term in inverted_index:
    inverted_index[term].sort(key=lambda x: x[0])

display_md(f"**Vocabulary size:** {len(inverted_index)} unique terms")

**Vocabulary size:** 80 unique terms

In [6]:
# Step 3: Show the three structures

# A) Vocabulary: sorted list of all terms
vocab = sorted(inverted_index.keys())
display_md(
    f"### A) Vocabulary ({len(vocab)} terms)\n\n"
    f"`{', '.join(vocab[:20])}` ..."
)

### A) Vocabulary (80 terms)

`a, adventure, algorithms, among, an, ancient, and, baker, bakery, ball, begin, beneath, beside, binary, birds, biscuits, bone, bread, bridges, cakes` ...

In [7]:
# B) Postings lists: for selected terms, show (doc_id, tf) pairs
selected_terms = ["cat", "dog", "forest", "adventure", "ancient"]
rows = []
for term in selected_terms:
    if term in inverted_index:
        postings = inverted_index[term]
        postings_str = ", ".join(f"({did}, tf={tf})" for did, tf in postings)
        rows.append([term, len(postings), postings_str])

display_md("### B) Postings Lists")
print_table(rows, headers=["Term", "DF", "Postings (doc_id, tf)"])

### B) Postings Lists

| Term      |   DF | Postings (doc_id, tf)                                                               |
|:----------|-----:|:------------------------------------------------------------------------------------|
| cat       |    5 | (b1, tf=2), (b10, tf=2), (b12, tf=1), (b4, tf=1), (b9, tf=1)                        |
| dog       |    6 | (b1, tf=2), (b10, tf=2), (b12, tf=1), (b3, tf=1), (b7, tf=4), (b9, tf=1)            |
| forest    |    7 | (b1, tf=1), (b10, tf=2), (b3, tf=1), (b5, tf=2), (b6, tf=1), (b8, tf=4), (b9, tf=1) |
| adventure |    3 | (b1, tf=1), (b11, tf=1), (b2, tf=1)                                                 |
| ancient   |    2 | (b2, tf=1), (b3, tf=1)                                                              |

In [8]:
# C) Document table: metadata for each document
rows = []
for doc_id, doc_text in documents():
    rows.append([doc_id, len(doc_tokens[doc_id]), doc_text[:50] + "..."])

display_md("### C) Document Table")
print_table(rows, headers=["Doc ID", "Length (tokens)", "Text (preview)"])

### C) Document Table

| Doc ID   |   Length (tokens) | Text (preview)                                        |
|:---------|------------------:|:------------------------------------------------------|
| b1       |                16 | The Cat and Dog in the Forest. A cat and a dog beg... |
| b2       |                12 | Cats of the Woodland. Wild cats begin an adventure... |
| b3       |                13 | The Forest Hound. A loyal dog follows a woodland t... |
| b4       |                11 | Feline Detective. A clever cat solves mysteries wi... |
| b5       |                15 | Random Forest for Pet Detection. A random forest m... |
| b6       |                14 | Forests of Search Trees. Algorithms explore a fore... |
| b7       |                15 | Dog Dog Dog! A dog chases a ball, finds a bone, an... |
| b8       |                14 | Forest Forest Forest! A forest guide names trees, ... |
| b9       |                 3 | Cat Dog Forest....                                    |
| b10      |                19 | Cat Dog Forest. A cat meets a dog in a forest besi... |
| b11      |                14 | Woodland Companions. A kitten and a puppy share a ... |
| b12      |                17 | The Pet Bakery. A cat, a dog, and a baker make cak... |

## Answering a Query: Inverted Lookup

Now finding documents for a term is instant: just look up its postings list.

In [9]:
query_term = "cat"
postings = inverted_index.get(query_term, [])

display_md(
    f"**Inverted index lookup for** `{query_term}`\n\n"
    f"- Postings accessed: **1 list** ({len(postings)} entries)\n"
    f"- Matches: {[doc_id for doc_id, tf in postings]}\n\n"
    f"Cost: O(1) lookup + O(postings_length) to read = O({len(postings)})\n\n"
    f"Compare with forward scan: O({len(forward_index)} x avg_length)"
)

**Inverted index lookup for** `cat`

- Postings accessed: **1 list** (5 entries)
- Matches: ['b1', 'b10', 'b12', 'b4', 'b9']

Cost: O(1) lookup + O(postings_length) to read = O(5)

Compare with forward scan: O(12 x avg_length)

## Boolean AND: Streaming Merge

Boolean AND requires documents that appear in ALL postings lists. Because
postings are sorted by doc_id, we can merge two lists with a two-pointer
algorithm in O(n + m) time, with no need to build sets.

In [10]:
def boolean_and_merge(postings_a, postings_b):
    """
    Merge two sorted postings lists to find common doc_ids.
    Returns list of doc_ids present in both lists.
    """
    result = []
    i, j = 0, 0
    steps = []  # Track pointer positions for visualization

    while i < len(postings_a) and j < len(postings_b):
        doc_a = postings_a[i][0]
        doc_b = postings_b[j][0]
        steps.append((i, j, doc_a, doc_b))

        if doc_a == doc_b:
            result.append(doc_a)
            i += 1
            j += 1
        elif doc_a < doc_b:
            i += 1
        else:
            j += 1

    return result, steps

display_md("**Algorithm:** Two-pointer merge on sorted postings lists.")

**Algorithm:** Two-pointer merge on sorted postings lists.

In [11]:
# AND query: "cat AND dog"
postings_cat = inverted_index.get("cat", [])
postings_dog = inverted_index.get("dog", [])

display_md(
    f"**Query:** `cat AND dog`\n\n"
    f"- `cat` postings: {[d for d, _ in postings_cat]}\n"
    f"- `dog` postings: {[d for d, _ in postings_dog]}"
)

**Query:** `cat AND dog`

- `cat` postings: ['b1', 'b10', 'b12', 'b4', 'b9']
- `dog` postings: ['b1', 'b10', 'b12', 'b3', 'b7', 'b9']

In [12]:
# Execute the merge and show pointer advancement
result, steps = boolean_and_merge(postings_cat, postings_dog)

rows = []
for step_num, (i, j, doc_a, doc_b) in enumerate(steps):
    action = "MATCH" if doc_a == doc_b else f"advance {'left' if doc_a < doc_b else 'right'}"
    rows.append([step_num + 1, f"i={i} -> {doc_a}", f"j={j} -> {doc_b}", action])

print_table(rows, headers=["Step", "Left pointer", "Right pointer", "Action"])
display_md(f"\n**Result:** `cat AND dog` = {result}")

|   Step | Left pointer   | Right pointer   | Action        |
|-------:|:---------------|:----------------|:--------------|
|      1 | i=0 -> b1      | j=0 -> b1       | MATCH         |
|      2 | i=1 -> b10     | j=1 -> b10      | MATCH         |
|      3 | i=2 -> b12     | j=2 -> b12      | MATCH         |
|      4 | i=3 -> b4      | j=3 -> b3       | advance right |
|      5 | i=3 -> b4      | j=4 -> b7       | advance left  |
|      6 | i=4 -> b9      | j=4 -> b7       | advance right |
|      7 | i=4 -> b9      | j=5 -> b9       | MATCH         |


**Result:** `cat AND dog` = ['b1', 'b10', 'b12', 'b9']

## Boolean OR: Union Merge

OR returns documents from either list. The merge walks both lists simultaneously,
collecting every doc_id exactly once.

In [13]:
def boolean_or_merge(postings_a, postings_b):
    """Merge two sorted postings lists to find the union of doc_ids."""
    result = []
    i, j = 0, 0

    while i < len(postings_a) and j < len(postings_b):
        doc_a = postings_a[i][0]
        doc_b = postings_b[j][0]

        if doc_a == doc_b:
            result.append(doc_a)
            i += 1
            j += 1
        elif doc_a < doc_b:
            result.append(doc_a)
            i += 1
        else:
            result.append(doc_b)
            j += 1

    # Remaining entries from whichever list is not exhausted
    while i < len(postings_a):
        result.append(postings_a[i][0])
        i += 1
    while j < len(postings_b):
        result.append(postings_b[j][0])
        j += 1

    return result

result_or = boolean_or_merge(postings_cat, postings_dog)
display_md(
    f"**Query:** `cat OR dog`\n\n"
    f"- Result: {result_or}\n"
    f"- Count: {len(result_or)} documents"
)

**Query:** `cat OR dog`

- Result: ['b1', 'b10', 'b12', 'b3', 'b4', 'b7', 'b9']
- Count: 7 documents

## Boolean NOT: Set Complement

NOT requires a list of all document IDs (from the document table) to compute
the complement.

In [14]:
def boolean_not(postings, all_doc_ids):
    """Return all doc_ids NOT in the postings list."""
    posting_ids = set(doc_id for doc_id, _ in postings)
    return [d for d in all_doc_ids if d not in posting_ids]

all_ids = sorted(ids())
result_not_cat = boolean_not(postings_cat, all_ids)
display_md(
    f"**Query:** `NOT cat`\n\n"
    f"- Documents WITHOUT 'cat': {result_not_cat}\n"
    f"- Count: {len(result_not_cat)} documents"
)

**Query:** `NOT cat`

- Documents WITHOUT 'cat': ['b11', 'b2', 'b3', 'b5', 'b6', 'b7', 'b8']
- Count: 7 documents

## Complex Query: (cat AND dog) OR forest

Real Boolean queries combine operators. We evaluate inside-out, reusing
our merge functions at each step.

In [15]:
# Step 1: Evaluate "cat AND dog"
postings_cat = inverted_index.get("cat", [])
postings_dog = inverted_index.get("dog", [])
cat_and_dog, _ = boolean_and_merge(postings_cat, postings_dog)

display_md(
    f"**Step 1:** `cat AND dog` = {cat_and_dog}"
)

**Step 1:** `cat AND dog` = ['b1', 'b10', 'b12', 'b9']

In [16]:
# Step 2: Get postings for "forest"
postings_forest = inverted_index.get("forest", [])
forest_ids = [doc_id for doc_id, _ in postings_forest]

display_md(
    f"**Step 2:** `forest` postings = {forest_ids}"
)

**Step 2:** `forest` postings = ['b1', 'b10', 'b3', 'b5', 'b6', 'b8', 'b9']

In [17]:
# Step 3: OR the two results together
# Convert cat_and_dog to postings format for reuse with our merge function
cat_dog_as_postings = [(d, 0) for d in cat_and_dog]
forest_as_postings = postings_forest

final_result = boolean_or_merge(cat_dog_as_postings, forest_as_postings)

display_md(
    f"**Step 3:** `(cat AND dog) OR forest` = {final_result}\n\n"
    f"| Sub-expression | Result |\n"
    f"|---|---|\n"
    f"| cat AND dog | {cat_and_dog} |\n"
    f"| forest | {forest_ids} |\n"
    f"| (cat AND dog) OR forest | {final_result} |"
)

**Step 3:** `(cat AND dog) OR forest` = ['b1', 'b10', 'b12', 'b3', 'b5', 'b6', 'b8', 'b9']

| Sub-expression | Result |
|---|---|
| cat AND dog | ['b1', 'b10', 'b12', 'b9'] |
| forest | ['b1', 'b10', 'b3', 'b5', 'b6', 'b8', 'b9'] |
| (cat AND dog) OR forest | ['b1', 'b10', 'b12', 'b3', 'b5', 'b6', 'b8', 'b9'] |

## Verifying the Results

Let us confirm our streaming merge produces correct results by checking
against the actual document texts.

In [18]:
# Verify: which documents actually contain both "cat" and "dog"?
rows = []
for doc_id in final_result:
    doc_text = text(doc_id)
    tokens = tokenize(doc_text)
    has_cat = "cat" in tokens
    has_dog = "dog" in tokens
    has_forest = "forest" in tokens
    reason = []
    if has_cat and has_dog:
        reason.append("cat AND dog")
    if has_forest:
        reason.append("forest")
    rows.append([doc_id, " OR ".join(reason), doc_text[:60]])

print_table(rows, headers=["Doc ID", "Matched via", "Text (preview)"])

| Doc ID   | Matched via           | Text (preview)                                               |
|:---------|:----------------------|:-------------------------------------------------------------|
| b1       | cat AND dog OR forest | The Cat and Dog in the Forest. A cat and a dog begin a woodl |
| b10      | cat AND dog OR forest | Cat Dog Forest. A cat meets a dog in a forest beside rivers, |
| b12      | cat AND dog           | The Pet Bakery. A cat, a dog, and a baker make cakes, bread, |
| b3       | forest                | The Forest Hound. A loyal dog follows a woodland trail throu |
| b5       | forest                | Random Forest for Pet Detection. A random forest model class |
| b6       | forest                | Forests of Search Trees. Algorithms explore a forest of bina |
| b8       | forest                | Forest Forest Forest! A forest guide names trees, flowers, r |
| b9       | cat AND dog OR forest | Cat Dog Forest.                                              |

## From Boolean to Ranking

Boolean retrieval returns unordered sets: a document either matches or it does
not. But users want ranked results. The good news: the same inverted index
supports both. Instead of intersecting postings, we accumulate scores from
each postings list. This is the topic of the next demo.

| Aspect | Boolean | Ranked |
|--------|---------|--------|
| Output | Set of doc IDs | Sorted list with scores |
| Postings use | Merge (AND/OR) | Score accumulation |
| User effort | Must craft precise query | Just type keywords |
| Index structure | Same inverted index | Same inverted index |

---

## Summary

| Concept | Key Point |
|---------|----------|
| Forward index | Doc -> terms; query requires full scan O(N) |
| Inverted index | Term -> docs; query reads only relevant postings |
| Vocabulary | Sorted dictionary mapping terms to postings pointers |
| Postings list | Sorted (doc_id, tf) pairs for one term |
| Document table | Doc_id -> metadata, length, source |
| AND merge | Two pointers, advance the smaller; O(n+m) |
| OR merge | Two pointers, collect both; O(n+m) |
| NOT | Complement against all doc IDs |
| Sorted postings | Enable streaming merge without building sets |

---

## Try It Yourself

1. Add a new term to the query. How does adding a rare term to an AND query
   affect the result size? Try `"cat AND ancient"`.

2. What happens with a term that does not exist in the vocabulary?
   Try looking up `"elephant"` in the inverted index.

3. Implement AND-NOT: find documents that contain "forest" but NOT "cat".
   Hint: merge "forest" postings against "cat" postings, skipping matches.

In [19]:
# Exercise: Implement AND-NOT merge
# Find documents with "forest" but not "cat"

# Your code here:
# def boolean_and_not_merge(postings_include, postings_exclude):
#     ...

# Verify with set operations:
forest_set = set(doc_id for doc_id, _ in inverted_index.get("forest", []))
cat_set = set(doc_id for doc_id, _ in inverted_index.get("cat", []))
display_md(f"**Expected result** (forest AND NOT cat): {sorted(forest_set - cat_set)}")

**Expected result** (forest AND NOT cat): ['b3', 'b5', 'b6', 'b8']